# Kochi-LLM — test the current checkpoint
Loads the newest checkpoint from the attached output (prefers `latest_backup_model.pth`, the most-trained weights) and generates from a few prompts.
Attach ONLY the latest Phase-2 output. Internet OFF, GPU T4 x1 is enough (generation fits on one GPU).

In [ ]:
%%bash
set -e
CAND=$(find /kaggle/input \( -name latest_backup_model.pth -o -name best_model.pth \) 2>/dev/null)
[ -z "$CAND" ] && { echo "No checkpoint attached"; ls -R /kaggle/input | head -40; exit 1; }
CKPT=""
T=0
while IFS= read -r f; do
  [ -z "$f" ] && continue
  t=$(stat -c %Y "$f")
  if [ "$t" -gt "$T" ]; then T=$t; CKPT=$f; fi
done <<EOF
$CAND
EOF
SRC=$(dirname "$CKPT")
echo "Using $CKPT"
PY_SRC=""
for d in /kaggle/input/datasets/dadhichisarker/kochi-llm/model "$SRC"; do
  if [ -f "$d/architecture.py" ]; then PY_SRC="$d"; break; fi
done
[ -z "$PY_SRC" ] && { echo "NO PY SOURCE"; exit 1; }
cp "$PY_SRC"/*.py /kaggle/working/
cp "$CKPT" /kaggle/working/test_model.pth
cp "$SRC"/custom_bpe_tokenizer.pkl /kaggle/working/
ls -lh /kaggle/working/*.py /kaggle/working/test_model.pth /kaggle/working/*.pkl


In [ ]:
import torch
from architecture import MyCustomLLM
from tokenizer import MyCustomTokenizer
from generate import generate

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ckpt = torch.load('test_model.pth', map_location='cpu', weights_only=False)
config = ckpt['config']
print(f"Checkpoint iter={ckpt.get('iter')} val={ckpt.get('val_loss')}")

model = MyCustomLLM(**config)
model.load_state_dict(ckpt['model_state_dict'])
model.to(device)
model.eval()
print(f"{sum(p.numel() for p in model.parameters())/1e6:.1f}M params on {device}")

tok = MyCustomTokenizer(vocab_size=config['vocab_size'])
tok.load('custom_bpe_tokenizer.pkl')
s = "Hello world, this is a tokenizer round-trip test."
assert tok.decode(tok.encode(s)) == s
print(f"Tokenizer round-trip: OK | vocab {tok.vocab_size}")


In [ ]:
prompts = [
    "The theory of relativity was developed by",
    "In computer science, a binary search algorithm is",
    "The capital city of France is",
    "Machine learning is a field of",
]
for p in prompts:
    out = generate(model, tok, p, max_new_tokens=100, temperature=0.8, top_k=40, device=device)
    print('='*70)
    print('PROMPT:', p)
    print(out)
